In [ ]:
from datetime import datetime, timedelta
import datetime as dt
from opendrift.models.oceandrift import OceanDrift
from opendrift.readers.reader_netCDF_CF_generic import Reader
import xarray as xr
import copernicusmarine
import geopandas as gpd
import sys
from multiprocessing import Pool
import glob

def concatenate_outputs(input_files, output_file):
    """
    Concatenate OpenDrift NetCDF outputs from multiple workers into one single output file.

    """

    ds_list = []
    trajectory_offset = 0
    for f in input_files:
            dsi = xr.open_dataset(f)
            traj_vars = [var for var in list(dsi.variables) if "trajectory" in dsi[var].dims]
            ds_traj = dsi[traj_vars]
            ds_traj = ds_traj.assign_coords(trajectory=ds_traj.trajectory + trajectory_offset)

            trajectory_offset += ds_traj.dims["trajectory"]
            ds_list.append(ds_traj)

    ds_conc = xr.concat(ds_list, dim="trajectory")
    ds_conc.to_netcdf(output_file)

    return ds_conc
    
farmID = '2'
year = '2017'
#infol = '/scratch/project_2018610/opendrift_outputs/simulation_15_part_per_km2/separate_depth/'
infol = '/scratch/project_2018610/opendrift_outputs/'
outfol = '/scratch/project_2018610/opendrift_outputs/simulation_15_part_per_km2/test/'
input_files = []
# for depth in [1, 5, 10]:
#     infile = 'simulation_farm_{}_z_-{}_June_September_{}.nc'.format(farmID, depth, year)
#     input_files.append(infol+infile)
for m in [7, 8]:
    infile = 'test_seeding_10000_every_third_day_region_1_2013_{}_test.nc'.format(m)
    input_files.append(infol+infile)

# outfile = 'simulation_farm_{}_June_September_{}.nc'.format(farmID, year)  
outfile = 'test_seeding_10000_every_third_day_region_1_2013_test.nc'
concatenate_outputs(input_files=input_files, output_file=outfol+outfile)

#ds = xr.open_dataset(infol+ncfile)